# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring a survey dataset on mental health indicators using the `mlcroissant` library. The data illustrates the application of AI-ready data standards and contains demographic and psychological assessment information.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
dataset_name = metadata.get('name', 'Dataset')
dataset_desc = metadata.get('description', '')
print(f"{dataset_name}: {dataset_desc}")

## 2. Data Overview
Review available record sets, fields, and their entity `@id`s.

Below, we enumerate the record sets available in the dataset, and list their fields and columns by `@id`. This is critical for referencing data unambiguously throughout your analysis.

In [ ]:
record_sets = dataset.record_sets

print('Available Record Sets:')
for rs in record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print("  Columns:")
    for column in rs.columns:
        print(f"    - {column.name} (@id: {column.id})")
    print()
if len(record_sets) > 0:
    print("Sample record from the first record set:")
    sample_records = list(dataset.records(record_set=record_sets[0].id))
    if len(sample_records) > 0:
        print(sample_records[0])
    else:
        print("No records found.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from all record sets
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
print(f"Record Set @ids: {record_set_ids}")

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records for record set with @id: {rs.id}")
    dataframes[rs.id] = df

# Display columns of the main record set
if len(record_sets) > 0:
    main_rs_id = record_sets[0].id
    print(f"Columns in record set {main_rs_id}: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Refer to the field and column `@id` values listed in the overview for precise references.

In [ ]:
# Example EDA: Analysis on GAD-7 scores
main_rs = record_sets[0] if len(record_sets) > 0 else None
main_rs_id = main_rs.id if main_rs is not None else None
df = dataframes.get(main_rs_id, pd.DataFrame())

# Identify the GAD-7 scores column by @id
gad7_id = None
for col in main_rs.columns:
    if 'GAD-7' in col.name or 'gad' in col.name.lower():
        gad7_id = col.id
        break
if gad7_id is None:
    gad7_id = df.columns[df.columns.str.contains('gad|GAD', case=False)].tolist()[0] if len(df.columns) > 0 else None

print(f"Using GAD-7 field @id: {gad7_id}")

# Filter records with GAD-7 scores above a threshold
threshold = 10
if gad7_id in df.columns:
    filtered_df = df[df[gad7_id] > threshold].copy()
    print(f"Filtered records with {gad7_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize GAD-7 scores
    filtered_df[f"{gad7_id}_normalized"] = (filtered_df[gad7_id] - filtered_df[gad7_id].mean()) / filtered_df[gad7_id].std()
    print(f"Normalized {gad7_id} for filtered records:")
    display(filtered_df[[gad7_id, f"{gad7_id}_normalized"].head()])

    # Group by 'village' column if available, using @id
    village_id = None
    for col in main_rs.columns:
        if 'village' in col.name.lower():
            village_id = col.id
            break
    if village_id and village_id in df.columns:
        grouped_df = filtered_df.groupby(village_id)[gad7_id].mean().reset_index()
        print(f"Mean GAD-7 score grouped by {village_id} (village):")
        display(grouped_df.head())
else:
    print(f"GAD-7 score field '{gad7_id}' not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Below is an example of visualizing normalized GAD-7 scores and their relationship with village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[gad7_id], bins=15, kde=True)
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot of GAD-7 by village (if available)
    if village_id and village_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[village_id], y=df[gad7_id])
        plt.xticks(rotation=45)
        plt.title('GAD-7 Scores by Village')
        plt.xlabel('Village')
        plt.ylabel('GAD-7 Score')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrates how to load and analyze a Croissant-enabled dataset using `mlcroissant`. Key fields and record sets are referenced by their `@id`, ensuring consistency. You can extend this notebook to explore additional fields, perform deeper statistical or causal analysis, and build predictive models using the processed data.